In [ ]:
%pip install numpy
%pip install numpy matplotlib seaborn scikit-learn
%pip install -U replay-trajectory-classification

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../Axona"))
sys.path.append(os.path.abspath(".."))
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from utils import prepare_data, maximum_a_posteriori_estimate
from replay_trajectory_classification import SortedSpikesClassifier #from the Frank lab: https://github.com/Eden-Kramer-Lab/replay_trajectory_classification
from replay_trajectory_classification import Environment, RandomWalk, Uniform, estimate_movement_var
import xarray as xr

c:\Users\ajifang\anaconda3\Lib\site-packages\replay_trajectory_classification\likelihoods\multiunit_likelihood.py:9: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Online decoding function
## Using decoding method from https://github.com/Eden-Kramer-Lab/replay_trajectory_classification

In [ ]:
filenames = [
             r"C:\decode\m20_alldays.mat",
            ] 

Env = 'R'  # VR or R
decoding_params = {
    'speedThres': 4,  # Speed threshold for running vs. immobile
    'upsampling_scale': 4, # Upsampling scale for position and spikes. 1 corresponds to 50 hz, 10 correspond to 500 hz...
    'nfold': 10, # n-fold cross validation
    'position_std': 3, #float or array_like, shape (n_position_dims,) Amount of smoothing for position when building the encoding model. Standard deviation of kernel.
    'place_bin_size': 2, #size of the place bin, bin number = env_size/place_bin_size
}


for filename in filenames:
    print(f'Processing file: {filename}')

    if filename == r"C:\decode\m20_alldays.mat":
        animalname = 'm20'
        Days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
        egf_channel_map = {**{day: [1] for day in Days}}
    else:
        raise ValueError("Unknown filename")
    
    Env = 'R'
    window_size = 100  # extraction window size in seconds
    celltype = None

    for session_id, day_id in enumerate(Days):
        print(f"Processing {animalname} Day {day_id}...")
        
        if celltype is not None:
            save_path = (
            f'/home/replay_decoding_'
            f'{celltype}/'
            f'{os.path.basename(filename).split(".")[0]}_{Env}_Day{day_id}_replay_results.pkl'
            )
        else:
            save_path = (
            f'/home/replay_decoding_R/'
            f'{os.path.basename(filename).split(".")[0]}_{Env}_Day{day_id}_replay_results.pkl'
            )
        
        
        if os.path.exists(save_path):
            print(f"Day {day_id} already processed, skipping...")
            continue
        

        speed, position, timestamps, SpikeArray, samplerate, reward_pos = prepare_data(filename=filename,
                                                                        session_id=session_id, 
                                                                        Env=Env,
                                                                        upsample_factor = 4,
                                                                        celltype=celltype)
        
        
        movement_var = estimate_movement_var(position, samplerate)
        place_bin_size  = 2
        environment = Environment(place_bin_size=place_bin_size)

        continuous_transition_types = [[RandomWalk(movement_var=(movement_var[0,0]+movement_var[1,1]/2)*10),  Uniform()],
                                        [Uniform(), Uniform()],
                                    ]

        classifier = SortedSpikesClassifier(
            environments=environment,
            continuous_transition_types=continuous_transition_types,
            sorted_spikes_algorithm='spiking_likelihood_kde_gpu',
            sorted_spikes_algorithm_params={'position_std': 3.0,
                                        }
        )

        speed_thres = 4 
        is_training = speed > speed_thres
        classifier.fit(position, SpikeArray.T, is_training=is_training)



        t = {'VR': 1, 'R': 2}.get(Env)
        channels = egf_channel_map[day_id]

        lfps = []
        for ch in channels:
            trial = get_trial(a=animalname, t=t, d=day_id, p='ZILONG', egf_channel=ch)
            lfps.append(trial.egf(bad_as=None))

        # stack into (n_samples, n_channels)
        lfps = np.vstack(lfps).T

        # sampling rate (same for all channels)
        egf_samplerate = trial.egf_samp_rate
        lfp_timestamps = np.arange(0, len(lfps)) / egf_samplerate
        speed4lfp = np.interp(lfp_timestamps, timestamps, speed)

        filtered_lfps = filter_ripple_band(lfps)
        ripple_times = Kay_ripple_detector(
            time=lfp_timestamps,
            filtered_lfps=filtered_lfps,
            speed=speed4lfp,
            sampling_frequency=egf_samplerate,
            speed_threshold=speed_thres, #maximum speed during ripple is 4 cm/s in R, or units/s in VR
            minimum_duration=0.02 #minimum duration of ripple event is 20 ms. Key et al used 15ms
        )

        display(ripple_times)
        
        print(len(ripple_times))
        
        N = len(timestamps)

        position_xr = xr.DataArray(position, dims=['time', 'position'], coords={'time': timestamps, 'position': ['x', 'y']})
        
        # create empty dataframe with desired columns
        replay_results = pd.DataFrame(columns=["ripple_start", "ripple_end", "duration", "ripple_position", "ripple_reward_position", "posterior", "MAP_est"])

        ripple_ind = 0
        for i in range(0, N//samplerate, window_size):
            start_sec = i
            end_sec = min(i + window_size, N // samplerate)
            
            print(f"Decoding from {start_sec}s to {end_sec}s")
            
            results = classifier.predict(SpikeArray.T[start_sec*samplerate:end_sec*samplerate, :], 
                                        time=timestamps[start_sec*samplerate:end_sec*samplerate], use_gpu=True) 
            
            ripple_events_in_window = ripple_times[
                (ripple_times['start_time'] >= start_sec) & 
                (ripple_times['end_time'] <= end_sec)
                ]
            
            for index, ripple_event in ripple_events_in_window.iterrows():
                ripple_start = ripple_event['start_time']
                ripple_end = ripple_event['end_time']
                
                #marginalize out state to get the posterior probability during the ripple event
                ripple_posterior = results.acausal_posterior.sel(time=slice(ripple_start, ripple_end)).sum('state')
                
                #compute the maximum a posterior estimate (MAP estimate)
                MAP_est = maximum_a_posteriori_estimate(ripple_posterior)

                #get position in ripple events using the position_xr
                ripple_position = position_xr.sel(time=slice(ripple_start, ripple_end))
                
                
                # append row
                if Env == 'VR':
                    replay_results.loc[ripple_ind] = {
                        "ripple_start": ripple_start,
                        "ripple_end": ripple_end,
                        "duration": ripple_end - ripple_start,
                        "ripple_position": ripple_position, # keep full xarray
                        "posterior": ripple_posterior,   # keep full xarray
                        "MAP_est": MAP_est
                    }
                elif Env == 'R':
                    replay_results.loc[ripple_ind] = {
                        "ripple_start": ripple_start,
                        "ripple_end": ripple_end,
                        "duration": ripple_end - ripple_start,
                        "ripple_position": ripple_position, # keep full xarray
                        "posterior": ripple_posterior,   # keep full xarray
                        "MAP_est": MAP_est
                    }
                
                ripple_ind += 1
                
        replay_results.to_pickle(save_path)   


            

In [ ]:
for file in filenames:
    print(f'Processing file: {filename}')

    if filename == r"C:\decode\m20_alldays.mat":
        animalname = 'm20'
        Days = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] 
        egf_channel_map = {**{day: [1] for day in Days}}
    else:
          
          
    for session_id, day_id in enumerate(Days):
        
        print(f'Processing Day {day_id}...')

        with open(f'/home/zilong/Desktop/DevDataAnalysis/Results/replay_decoding/'
                f'{os.path.basename(filename).split(".")[0]}_VR_Day{Days[session_id]}_replay_results.pkl', 'rb') as f:
            replay_results = pickle.load(f)

        # define fixed grid in aligned coordinates
        grid_x = np.linspace(-10, 10, 20)
        grid_y = np.linspace(-10, 10, 20)
        GX, GY = np.meshgrid(grid_x, grid_y)

        Z_total = np.zeros_like(GX)

        all_map_trajs = [] 
        All_MAP_est = [] 
        
        for idx in range(len(replay_results)):

            posterior_i = replay_results.loc[idx]['posterior']
            MAP_est_i = replay_results.loc[idx]['MAP_est']
            All_MAP_est.append(MAP_est_i)
            # transform MAP estimates
            map_coords = np.vstack([MAP_est_i.x_position.values,
                                    MAP_est_i.y_position.values]).T   # shape (n_time, 2)
            
            center = map_coords[0,:]
            # apply transform
            map_centered = map_coords - center

            all_map_trajs.append({
                "x": map_centered[:,0],
                "y": map_centered[:,1],
                "t": MAP_est_i['time'].values
            })

            # transform posterior grida
            posterior_xy = posterior_i.sum('time').transpose('y_position', 'x_position')
            X, Y = np.meshgrid(posterior_xy['x_position'].values,
                            posterior_xy['y_position'].values, indexing='xy')
            coords = np.stack([X.ravel(), Y.ravel()], axis=1)
            coords_centered = coords - center
            Z = posterior_xy.values.ravel()

            # interpolate onto fixed grid
            Z_interp = griddata(coords_centered, Z, (GX, GY), method='nearest', fill_value=np.nan)
            Z_total += np.nan_to_num(Z_interp)
            
        replay_diffusion, replay_intercept, time_bins, ave_steps = utils.get_diffusion_exponent(All_MAP_est, stepnums=15)
        
        results[day_id] = {
            'replay_scale': replay_intercept,
            'replay_exp': replay_diffusion,
            'sum_trajs': all_map_trajs,
            'sum_posterior': Z_total
        }